In [12]:
import sys; sys.path.insert(0, '..')
import pandas as pd
import numpy as np

from src.ingest.rama import load_wide, to_long

long = to_long(load_wide('../data/raw/2025O3.xls'), 'O3')

# pivot: convierte formato largo a matriz horas x estaciones
matriz = long.pivot(index='timestamp', columns='station', values='value')

# Excluir las que no operaron en 2025: no aportan señal ni ausencia informativa
vacias = matriz.columns[matriz.notna().sum() == 0].tolist()
print(f"Sin dato en todo 2025: {vacias}")
matriz = matriz.drop(columns=vacias)

n_est = matriz.shape[1]
print(f"Matriz resultante: {matriz.shape[0]} horas x {n_est} estaciones\n")

faltantes = matriz.isna().sum(axis=1)

print("Estaciones faltantes por hora:")
print(f"  media:    {faltantes.mean():.1f} de {n_est}")
print(f"  mediana:  {faltantes.median():.0f}")
print(f"  mínimo:   {faltantes.min()}")
print(f"  máximo:   {faltantes.max()}")
print(f"\n  % de información disponible en la hora típica: "
      f"{(1 - faltantes.median()/n_est)*100:.1f}%")

print("\nDistribución (% de horas con N faltantes):")
dist = (faltantes.value_counts().sort_index() / len(matriz) * 100).round(2)
print(dist.head(20).to_string())

Sin dato en todo 2025: ['COY', 'SFE', 'SJA']
Matriz resultante: 8760 horas x 33 estaciones

Estaciones faltantes por hora:
  media:    8.0 de 33
  mediana:  7
  mínimo:   0
  máximo:   33

  % de información disponible en la hora típica: 78.8%

Distribución (% de horas con N faltantes):
0      0.43
1      3.13
2      5.06
3      6.47
4      9.35
5      7.50
6     13.44
7     11.79
8      9.09
9      6.13
10     3.85
11     1.74
12     3.41
13     4.84
14     4.08
15     2.76
16     3.32
17     1.10
18     0.27
19     0.10


In [13]:
# 1. ¿Cuántas horas son catastróficas?
for umbral in [15, 20, 25, 30, 33]:
    n = (faltantes >= umbral).sum()
    print(f"Horas con >={umbral} faltantes: {n:>4} ({n/len(matriz)*100:.2f}%)")

Horas con >=15 faltantes:  850 (9.70%)
Horas con >=20 faltantes:  188 (2.15%)
Horas con >=25 faltantes:  183 (2.09%)
Horas con >=30 faltantes:  132 (1.51%)
Horas con >=33 faltantes:   18 (0.21%)


In [14]:
# 2. ¿Se concentran en alguna hora del día?
df_f = pd.DataFrame({'faltantes': faltantes})
df_f['hora'] = df_f.index.hour
print("\nFaltantes promedio por hora del día:")
print(df_f.groupby('hora')['faltantes'].mean().round(1).to_string())


Faltantes promedio por hora del día:
hora
0     11.0
1     11.0
2     11.0
3      7.3
4      7.2
5      7.3
6      7.3
7      7.4
8      7.4
9      7.5
10     7.8
11     8.1
12     8.3
13     8.1
14     8.0
15     7.8
16     7.7
17     7.6
18     7.5
19     7.5
20     7.4
21     7.4
22     7.4
23     7.6


In [15]:
# 3. ¿Son eventos aislados o bloques largos?
critico = (faltantes >= 25).astype(int)
grupos = (critico != critico.shift()).cumsum()
rachas = critico.groupby(grupos).sum()
rachas = rachas[rachas > 0]
print(f"\nEventos con >=25 faltantes: {len(rachas)}")
print(f"Duración: media {rachas.mean():.1f} h, máx {rachas.max()} h")


Eventos con >=25 faltantes: 61
Duración: media 3.0 h, máx 3 h


In [16]:
from src.detect.windows import construir_ventanas

# Split cronologico ANTES de normalizar
corte = int(len(matriz) * 0.8)
mat_tr, mat_te = matriz.iloc[:corte], matriz.iloc[corte:]

Xtr, ytr, ts_tr, mu, sigma = construir_ventanas(mat_tr, 'CCA', ventana=24)
Xte, yte, ts_te, _, _ = construir_ventanas(mat_te, 'CCA', ventana=24,
                                           mu=mu, sigma=sigma)

print(f"Entrenamiento: {Xtr.shape}   y: {ytr.shape}")
print(f"Prueba:        {Xte.shape}   y: {yte.shape}")
print(f"\nRango temporal train: {ts_tr[0]} -> {ts_tr[-1]}")
print(f"Rango temporal test:  {ts_te[0]} -> {ts_te[-1]}")
print(f"\nFraccion de datos reales en X (media de la mascara): "
      f"{Xtr[:, :, 32:].mean():.3f}")

ValueError: too many values to unpack (expected 5, got 6)

In [17]:
from src.detect.lstm import construir_modelo, entrenar

modelo = construir_modelo(ventana=24, n_features=Xtr.shape[2])
modelo.summary()

I0000 00:00:1790130200.648880  107120 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790130200.649505  107120 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790130200.672719  107120 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790130201.583513  107120 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

NameError: name 'Xtr' is not defined

In [ ]:
historia = entrenar(modelo, Xtr, ytr, val_frac=0.2, epocas=50)

## Comparacion entre datos predecidos y datos reales

In [ ]:
import matplotlib.pyplot as plt

pred = modelo.predict(Xte, verbose=0).flatten()
residuo = yte - pred

df_ev = pd.DataFrame({
    'timestamp': ts_te,
    'real': yte,
    'predicho': pred,
    'residuo': residuo,
})
df_ev['hora'] = pd.to_datetime(df_ev.timestamp).dt.hour

print(f"MAE  test: {np.abs(residuo).mean():.2f} ppb")
print(f"Sesgo:     {residuo.mean():+.2f} ppb")
print(f"σ residuo: {residuo.std():.2f} ppb")
print(f"\nPercentiles del |residuo|:")
for p in [50, 75, 90, 95, 99]:
    print(f"  p{p}: {np.percentile(np.abs(residuo), p):6.2f} ppb")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

n = 336   # dos semanas
ax1.plot(df_ev.timestamp[:n], df_ev.real[:n], lw=1.5, label='Real')
ax1.plot(df_ev.timestamp[:n], df_ev.predicho[:n], lw=1.5, alpha=0.8,
         label='Predicho')
ax1.set_ylabel('O₃ (ppb)')
ax1.legend()
ax1.grid(alpha=0.3)
ax1.set_title('Predicción de CCA a partir de 32 estaciones vecinas')

ax2.plot(df_ev.timestamp[:n], df_ev.residuo[:n], lw=1, color='crimson')
ax2.axhline(0, c='gray', lw=0.8)
ax2.set_ylabel('Residuo (ppb)')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/img/lstm_prediccion.png', dpi=150)
plt.show()

In [ ]:
por_hora = df_ev.groupby('hora').agg(
    real_media=('real', 'mean'),
    mae=('residuo', lambda r: r.abs().mean()),
    sesgo=('residuo', 'mean'),
).round(2)
print(por_hora.to_string())

In [ ]:
import importlib
import src.detect.windows
importlib.reload(src.detect.windows)
from src.detect.windows import construir_ventanas

Xtr, ytr, ts_tr, mu, sigma = construir_ventanas(mat_tr, 'CCA', ventana=24)
Xte, yte, ts_te, _, _ = construir_ventanas(mat_te, 'CCA', ventana=24,
                                           mu=mu, sigma=sigma)

print(f"Nueva forma: {Xtr.shape}")   # debe ser (6764, 24, 66)

modelo2 = construir_modelo(ventana=24, n_features=Xtr.shape[2])
historia2 = entrenar(modelo2, Xtr, ytr, val_frac=0.2, epocas=50, verbose=0)

pred2 = modelo2.predict(Xte, verbose=0).flatten()
res2 = yte - pred2

df2 = pd.DataFrame({'timestamp': ts_te, 'real': yte,
                    'predicho': pred2, 'residuo': res2})
df2['hora'] = pd.to_datetime(df2.timestamp).dt.hour

print(f"\nMAE test: {np.abs(res2).mean():.2f} ppb   (antes 8.39)")
print(f"Sesgo:    {res2.mean():+.2f} ppb   (antes +2.80)")

comp = pd.DataFrame({
    'real_media': df2.groupby('hora').real.mean().round(1),
    'mae_antes': df_ev.groupby('hora').residuo.apply(lambda r: r.abs().mean()).round(2),
    'mae_ahora': df2.groupby('hora').residuo.apply(lambda r: r.abs().mean()).round(2),
    'sesgo_ahora': df2.groupby('hora').residuo.mean().round(2),
})
print(comp.to_string())

In [ ]:
import importlib, src.detect.windows
importlib.reload(src.detect.windows)
from src.detect.windows import construir_ventanas, perfil_horario

# Perfil SOLO del entrenamiento
perf = perfil_horario(mat_tr['CCA'])
print("Perfil horario de CCA (train):")
print(perf.round(1).to_string())

Xtr3, ytr3, ts_tr3, Btr3, mu3, sg3 = construir_ventanas(
    mat_tr, 'CCA', ventana=24, perfil=perf, predecir_desviacion=True)
Xte3, yte3, ts_te3, Bte3, _, _ = construir_ventanas(
    mat_te, 'CCA', ventana=24, mu=mu3, sigma=sg3,
    perfil=perf, predecir_desviacion=True)

print(f"\ny ahora es desviacion: media {ytr3.mean():+.2f}, "
      f"rango [{ytr3.min():.1f}, {ytr3.max():.1f}]")

modelo3 = construir_modelo(ventana=24, n_features=Xtr3.shape[2])
historia3 = entrenar(modelo3, Xtr3, ytr3, val_frac=0.2, epocas=60, verbose=0)

# Reconstruir el valor absoluto
desv_pred = modelo3.predict(Xte3, verbose=0).flatten()
pred3 = Bte3 + desv_pred
real3 = Bte3 + yte3
res3 = real3 - pred3

print(f"\nMAE test: {np.abs(res3).mean():.2f} ppb   "
      f"(v1: 8.39, v2: 7.62)")
print(f"Sesgo:    {res3.mean():+.2f} ppb")

df3 = pd.DataFrame({'timestamp': ts_te3, 'real': real3,
                    'predicho': pred3, 'residuo': res3})
df3['hora'] = pd.to_datetime(df3.timestamp).dt.hour

comp = pd.DataFrame({
    'real': df3.groupby('hora').real.mean().round(1),
    'mae_v2': df2.groupby('hora').residuo.apply(lambda r: r.abs().mean()).round(2),
    'mae_v3': df3.groupby('hora').residuo.apply(lambda r: r.abs().mean()).round(2),
    'sesgo_v3': df3.groupby('hora').residuo.mean().round(2),
})
print(comp.to_string())

### Intento 3: perfil horario adaptativo

**Diagnóstico previo.** El perfil calculado sobre el conjunto de entrenamiento
(ene–oct) no describe el periodo de prueba (nov–dic):

| Hora | Perfil ene–oct | Real nov–dic | Desfase |
|---|---|---|---|
| 0 | 17.2 | 11.4 | −5.8 |
| 7 | 8.3 | 4.8 | −3.5 |
| 16 | 67.4 | 78.0 | **+10.6** |
| 17 | 55.8 | 63.9 | +8.1 |

El error no proviene del modelo sino de la línea base: aunque predijera la
desviación con exactitud, el valor reconstruido arrastraría el desfase del
perfil.

**Propuesta.** Sustituir el promedio anual fijo por una media móvil por hora de
los últimos N días. La línea base sigue al régimen vigente en lugar de
representar un promedio que no corresponde a ningún periodo concreto.

**Justificación operativa.** Es lo que haría un sistema desplegado: recalibrar
continuamente contra el comportamiento reciente. Un perfil fijo calculado en
enero tendría el mismo desfase en abril, con split cronológico o sin él.

**Criterio de éxito.** Que el sesgo en las horas 15–17 descienda de +13 a +15
ppb hasta ±3 ppb. Sin esa corrección, la evaluación sobre datos atacados no es
interpretable: un residuo de 17 ppb sin ataque enmascara por completo un ataque
del bit 4 (16 ppb).

**Parámetro a barrer.** La ventana N (7, 14, 30 días) es un compromiso: muy
corta sigue el ruido diario, muy larga no reacciona al cambio estacional.

In [ ]:
import importlib, src.detect.windows
importlib.reload(src.detect.windows)
from src.detect.windows import construir_ventanas, perfil_adaptativo

# El perfil se calcula sobre la serie COMPLETA de forma causal:
# cada punto usa solo su propio pasado, por lo que no hay fuga.
serie_cca = matriz['CCA']
perf_ad = perfil_adaptativo(serie_cca, dias=14)

corte = int(len(matriz) * 0.8)
perf_tr, perf_te = perf_ad.iloc[:corte], perf_ad.iloc[corte:]

Xtr4, ytr4, ts_tr4, Btr4, mu4, sg4 = construir_ventanas(
    mat_tr, 'CCA', ventana=24, perfil=perf_tr, predecir_desviacion=True)
Xte4, yte4, ts_te4, Bte4, _, _ = construir_ventanas(
    mat_te, 'CCA', ventana=24, mu=mu4, sigma=sg4,
    perfil=perf_te, predecir_desviacion=True)

modelo4 = construir_modelo(ventana=24, n_features=Xtr4.shape[2])
historia4 = entrenar(modelo4, Xtr4, ytr4, val_frac=0.2, epocas=60, verbose=0)

pred4 = Bte4 + modelo4.predict(Xte4, verbose=0).flatten()
real4 = Bte4 + yte4
res4 = real4 - pred4

print(f"MAE test: {np.abs(res4).mean():.2f} ppb   "
      f"(v1 8.39 | v2 7.62 | v3 7.92)")
print(f"Sesgo:    {res4.mean():+.2f} ppb")

df4 = pd.DataFrame({'timestamp': ts_te4, 'real': real4,
                    'predicho': pred4, 'residuo': res4})
df4['hora'] = pd.to_datetime(df4.timestamp).dt.hour

comp = pd.DataFrame({
    'real': df4.groupby('hora').real.mean().round(1),
    'mae_v2': df2.groupby('hora').residuo.apply(lambda r: r.abs().mean()).round(2),
    'mae_v3': df3.groupby('hora').residuo.apply(lambda r: r.abs().mean()).round(2),
    'mae_v4': df4.groupby('hora').residuo.apply(lambda r: r.abs().mean()).round(2),
    'sesgo_v4': df4.groupby('hora').residuo.mean().round(2),
})
print(comp.to_string())

In [ ]:
print(f"Perfil adaptativo: {perf_ad.isna().sum()} NaN de {len(perf_ad)} "
      f"({perf_ad.isna().mean()*100:.1f}%)")
print(f"Primeros valores válidos: {perf_ad.first_valid_index()}")
print(f"\nBte4: {np.isnan(Bte4).sum()} NaN de {len(Bte4)}")
print(f"ytr4: {np.isnan(ytr4).sum()} NaN de {len(ytr4)}")

In [ ]:
import importlib
import src.detect.windows
importlib.reload(src.detect.windows)
from src.detect.windows import perfil_adaptativo, construir_ventanas, perfil_horario

In [ ]:
for dias in [7, 14, 21, 30, 45]:
    perf = perfil_adaptativo(serie_cca, dias=dias)
    pt, pv = perf.iloc[:corte], perf.iloc[corte:]
    
    Xa, ya, tsa, Ba, mua, sga = construir_ventanas(
        mat_tr, 'CCA', ventana=24, perfil=pt, predecir_desviacion=True)
    Xb, yb, tsb, Bb, _, _ = construir_ventanas(
        mat_te, 'CCA', ventana=24, mu=mua, sigma=sga,
        perfil=pv, predecir_desviacion=True)
    
    m = construir_modelo(24, Xa.shape[2])
    entrenar(m, Xa, ya, epocas=40, verbose=0)
    
    r = (Bb + yb) - (Bb + m.predict(Xb, verbose=0).flatten())
    print(f"dias={dias:>2}:  MAE {np.abs(r).mean():5.2f}  "
          f"sesgo {r.mean():+5.2f}  σ {r.std():5.2f}")

# Qué pasó con el modelo, explicado desde cero

Esta celda documenta el proceso de los cuatro intentos, los conceptos
involucrados y por qué la versión con peor número es la que sirve.

---

## 1. Qué está haciendo el modelo

La idea, en una frase:

> Usando lo que midieron las otras 32 estaciones de la ciudad durante las
> últimas 24 horas, adivina cuánto ozono está midiendo CCA en este momento.

**CCA no aparece en la entrada.** El modelo nunca la ve. Tiene que deducirla
mirando a sus vecinas.

Por ejemplo, el 15 de marzo a las 14:00:

```
Lo que el modelo recibe:
   qué midieron ACO, AJM, TLA, PED... (32 estaciones)
   desde el 14 a las 14:00 hasta el 15 a las 13:00

Lo que tiene que responder:
   ¿cuánto midió CCA el 15 a las 14:00?

La respuesta correcta (que el modelo no ve):
   84 ppb
```

### Por qué esto sirve para detectar ataques

Si el atacante manipula la lectura de CCA, **las vecinas siguen diciendo la
verdad**. El modelo predice lo que CCA *debería* estar midiendo, y si el valor
que llega es muy distinto, algo pasó.

```
Modelo predice (según las vecinas):   84 ppb
Valor que llega al servidor:         116 ppb
                                     ───────
Diferencia:                           32 ppb   ← sospechoso
```

Esa diferencia se llama **residuo**, y es todo el detector.

---

## 2. Los conceptos, con ejemplos

### Residuo

La diferencia entre lo que pasó de verdad y lo que el modelo predijo.

```
residuo = valor_real − valor_predicho
```

Si CCA midió 84 y el modelo predijo 80, el residuo es **+4**.
Si CCA midió 84 y el modelo predijo 90, el residuo es **−6**.

El signo importa: positivo significa que el modelo se quedó corto; negativo,
que se pasó.

### MAE — error absoluto medio

*Mean Absolute Error*. Responde: **¿de cuánto se equivoca el modelo, en
promedio?**

«Absoluto» significa que se ignora el signo: un error de −6 cuenta igual que uno
de +6. Solo importa el tamaño.

Ejemplo con cinco predicciones:

| Real | Predicho | Residuo | Sin signo |
|---|---|---|---|
| 84 | 80 | +4 | 4 |
| 62 | 70 | −8 | 8 |
| 15 | 13 | +2 | 2 |
| 90 | 85 | +5 | 5 |
| 45 | 51 | −6 | 6 |

```
MAE = (4 + 8 + 2 + 5 + 6) / 5 = 5 ppb
```

**El modelo se equivoca 5 ppb en promedio.** Se lee en las mismas unidades que
los datos, por eso es la métrica más interpretable.

### Sesgo

**¿El modelo se equivoca siempre hacia el mismo lado?**

Es el promedio de los residuos, pero **conservando el signo**:

```
Sesgo = (+4 − 8 + 2 + 5 − 6) / 5 = −0.6 ppb
```

Casi cero: se equivoca a veces hacia arriba, a veces hacia abajo, y se
compensan. Eso es lo deseable.

Ahora compara con este caso:

| Real | Predicho | Residuo |
|---|---|---|
| 84 | 70 | +14 |
| 90 | 75 | +15 |
| 78 | 62 | +16 |
| 82 | 69 | +13 |

```
MAE   = 14.5 ppb
Sesgo = +14.5 ppb    ← idéntico al MAE
```

**Todos los errores van en la misma dirección.** El modelo siempre predice de
menos. Eso no es ruido, es un defecto sistemático.

**Regla práctica:** cuando el MAE y el sesgo son casi iguales, el modelo tiene
un problema estructural. Cuando el sesgo está cerca de cero, el error es ruido
aleatorio.

### σ (sigma) — desviación estándar

La letra griega sigma. Mide **qué tanto se dispersan los valores alrededor de su
promedio**.

Ejemplo con dos modelos, ambos con sesgo cero:

```
Modelo A — residuos:   +2, −1, +1, −2, 0
Modelo B — residuos:  +30, −28, +25, −27, 0
```

Los dos promedian cero. Pero el A se equivoca poco y el B muchísimo.

σ captura esa diferencia: σ del A ≈ 1.6, σ del B ≈ 27.

**σ pequeña = predicciones consistentes. σ grande = predicciones erráticas.**

Para el detector, σ es lo que importa: es el «ruido de fondo» que un ataque
tiene que superar para notarse.

---

## 3. El problema que apareció

El primer modelo dio un MAE de 8.39 ppb. Suena aceptable. Pero al mirar hora por
hora:

| Hora | O₃ real | MAE | Sesgo |
|---|---|---|---|
| 4:00 | 8.8 | 5.4 | −3.0 |
| 12:00 | 71.2 | 8.4 | +7.6 |
| **16:00** | **78.0** | **20.0** | **+18.7** |
| 22:00 | 13.5 | 5.0 | +0.2 |

A las 4 de la mañana se equivoca 5 ppb. **A las 4 de la tarde, 20 ppb.**

Y fíjate en las 16:00: MAE de 20.0 y sesgo de +18.7, prácticamente iguales.
Aplicando la regla de arriba, eso es un defecto sistemático.

### Qué significa «sesgo de +18.7»

El modelo **siempre predice de menos** a esa hora.

```
CCA midió 78 ppb  →  el modelo predijo 59
CCA midió 82 ppb  →  el modelo predijo 64
CCA midió 75 ppb  →  el modelo predijo 57
```

Nunca al revés. Se queda corto todas las veces.

### Por qué pasa: el modelo aplana la curva

El ozono tiene un ciclo diario muy marcado: casi nada de madrugada, mucho por la
tarde.

```
Real:      ___/‾‾‾‾\___     sube alto y baja
Predicho:  __/‾‾‾‾‾\__      sube a medias
```

El modelo aprende que «normalmente hay poco ozono» —lo cual es cierto el 77% del
tiempo— y por eso nunca se atreve a predecir los picos.

Es como un pronosticador que dice «mañana no llueve» todos los días. Acierta
casi siempre, y falla justo cuando importa.

### Por qué esto rompe el detector

El detector marca anomalía cuando el residuo es grande. Pero a las 16:00 el
residuo ya vale +18.7 **sin ningún ataque**.

```
Sin ataque a las 16:00:        residuo = +18.7
Con ataque del bit 4 (+16):    residuo = +34.7
```

Para no marcar todas las horas normales como ataques, el umbral tendría que
estar por encima de 18.7. Y entonces el ataque de 16 ppb apenas asoma.

**Y las 15:00–17:00 son precisamente la ventana donde el ataque es más
efectivo.** El detector estaría ciego justo donde más se necesita.

---

## 4. Los cuatro intentos

### v1 — Modelo base

Solo las 32 vecinas y su máscara de validez.

```
MAE: 8.39    Sesgo a las 16:00: +18.68
```

Funciona en general, pero aplana el pico.

### v2 — Añadir la hora del día

Se le dice al modelo qué hora es, codificada como seno y coseno para que las
23:00 y las 00:00 queden juntas (son consecutivas, no opuestas).

Razón física: el ozono se forma con la luz solar, así que depende directamente
de la hora.

```
MAE: 7.62 (mejor)    Sesgo a las 16:00: +17.51 (casi igual)
```

**Mejoró el promedio, no el problema.** La ganancia se concentró en las horas de
madrugada, donde el ciclo es predecible. El pico siguió aplanado.

### v3 — Predecir la desviación en lugar del valor

En lugar de pedirle «predice 78 ppb», se le pide «predice cuánto se desvía de lo
normal para esta hora».

```
Lo normal a las 16:00 (promedio del año):  67.4 ppb
El modelo predice la desviación:           +10.6
Valor final:                       67.4 + 10.6 = 78 ppb
```

La idea: el ciclo diario se puede calcular con un promedio simple, sin
necesidad de una red neuronal. Al restarlo, el modelo solo aprende la parte
impredecible.

```
MAE: 7.92    Sesgo a las 16:00: +13.95
```

**Mejoró algo, pero no lo suficiente.** Y al investigar por qué, apareció la
causa real.

---

## 5. La causa real: el promedio estaba desactualizado

El modelo entrena con datos de enero a octubre y se evalúa en noviembre y
diciembre. El promedio por hora se calculó con los meses de entrenamiento.

Pero el ozono **no se comporta igual en todos los meses**:

| Hora | Promedio ene–oct | Real nov–dic | Desfase |
|---|---|---|---|
| 0:00 | 17.2 | 11.4 | −5.8 |
| 7:00 | 8.3 | 4.8 | −3.5 |
| **16:00** | **67.4** | **78.0** | **+10.6** |
| 17:00 | 55.8 | 63.9 | +8.1 |

En noviembre el pico de la tarde es **10 ppb más alto** que el promedio anual.

Entonces el modelo hace esto:

```
Base que se le da:              67.4 ppb   (promedio ene-oct)
Desviación que predice:         +10.6
Valor que produce:              78.0
Valor real de noviembre:        88.0
                                ──────
Error:                          −10.0      ← viene de la base, no del modelo
```

**Aunque predijera la desviación perfectamente, el resultado estaría 10 ppb
abajo**, porque la base de la que parte es de otra estación del año.

### Esto explica tres cosas que veníamos arrastrando

Lo que parecían problemas independientes son el mismo fenómeno:

1. El split estratificado daba peores resultados que el cronológico
2. La tasa de falsos positivos varía de 1.10% a 10.42% según el mes
3. El error en test (8.39) era peor que en validación (6.86)

**Todo es deriva estacional.** El comportamiento del ozono cambia a lo largo del
año, y cualquier referencia fija queda desactualizada.

---

## 6. v4 — Perfil adaptativo

En lugar de un promedio fijo de todo el año, se usa el promedio de **los últimos
14 días**.

```
Promedio fijo:        siempre 67.4 para las 16:00
Promedio adaptativo:  en marzo 61, en julio 70, en noviembre 76
```

La base se va actualizando sola. Es lo que haría un sistema real: recalibrarse
continuamente contra el comportamiento reciente.

**Detalle importante:** el promedio se calcula solo con datos **anteriores** al
momento que se predice. Si usara el día que está prediciendo, sería hacer
trampa.

### El resultado

| Hora | Sesgo v3 | Sesgo v4 |
|---|---|---|
| 15:00 | +12.88 | **−2.35** |
| 16:00 | +13.95 | **−1.72** |
| 17:00 | +10.84 | **−1.01** |
| Global | +3.62 | **+0.18** |

**El sesgo desapareció.** De +14 a menos de ±3.

Pero el MAE subió a 9.35 — el peor de los cuatro.

---

## 7. Por qué el peor número es el mejor modelo

Parece contradictorio, pero no lo es.

### El error cambió de naturaleza

**Antes (v3):** el modelo se equivocaba **siempre hacia abajo** a las 16:00.

```
residuos: +14, +15, +13, +16, +14, +15...
```

**Ahora (v4):** se equivoca en ambas direcciones.

```
residuos: +18, −17, +20, −16, +19, −18...
```

El MAE es parecido. Pero son situaciones completamente distintas para un
detector.

### Con sesgo sistemático, el detector no puede funcionar

```
v3, a las 16:00:
   Sin ataque:              residuo ≈ +14
   Con ataque del bit 4:    residuo ≈ +30

El umbral debe superar 14 para no marcar horas normales.
Solo quedan 16 puntos de margen.
```

### Con ruido aleatorio, sí puede

```
v4, a las 16:00:
   Sin ataque:              residuo ≈ 0, con dispersión de ±19
   Con ataque del bit 4:    residuo ≈ +16
   Con ataque del bit 5:    residuo ≈ +32
```

El umbral se pone alrededor de cero más unas cuantas desviaciones. Los ataques
grandes salen del rango normal.

**Un sesgo sistemático es ceguera. El ruido es falta de sensibilidad.** El
segundo se puede trabajar; el primero no.

---

## 8. El barrido de la ventana

Se probaron distintas longitudes para el promedio móvil:

| Días | MAE | Sesgo | σ |
|---|---|---|---|
| 7 | 9.77 | +0.96 | 13.42 |
| **14** | **9.13** | **−1.19** | **12.77** |
| 21 | 10.31 | +2.00 | 13.93 |
| 30 | 9.20 | +1.75 | 12.32 |
| 45 | 9.13 | +3.30 | 12.00 |

**El resultado es plano:** todo entre 9.1 y 10.3. La longitud de la ventana no
es el factor determinante.

Se observa un compromiso: ventanas largas dan σ menor (base más estable) pero
sesgo mayor (menos reactiva al cambio estacional).

**Se adopta 14 días**: mejor sesgo, σ competitiva.

---

## 9. Dónde queda el detector

Situación actual:

```
Ruido de fondo del modelo:   σ ≈ 13 ppb en general
                             σ ≈ 19 ppb en el pico vespertino
```

Un ataque tiene que superar ese ruido para notarse:

| Ataque | Δ | Contra σ=13 | Contra σ=19 (pico) |
|---|---|---|---|
| bit 3 | 8 ppb | 0.6× — invisible | 0.4× — invisible |
| bit 4 | 16 ppb | 1.2× — dudoso | 0.8× — invisible |
| bit 5 | 32 ppb | **2.5× — detectable** | 1.7× — dudoso |
| bit 6 | 64 ppb | **4.9× — claro** | **3.4× — claro** |

Los bits 5 y 6 —la zona explotable identificada en el notebook 06— deberían ser
detectables.

### Lo que falta

Evaluar sobre datos realmente atacados. Hasta ahora todo se ha hecho con datos
limpios: solo se ha medido qué tan bien predice el modelo, no si detecta.

**Ese es el siguiente paso**, y es el que dice si todo esto sirvió.

---

## 10. Resumen

| Versión | Cambio | MAE | Sesgo 16h | Veredicto |
|---|---|---|---|---|
| v1 | Base | 8.39 | +18.68 | Ciego en la tarde |
| v2 | + hora del día | 7.62 | +17.51 | Ciego en la tarde |
| v3 | Predice desviación | 7.92 | +13.95 | Ciego en la tarde |
| **v4** | **Perfil adaptativo** | 9.35 | **−1.72** | **Utilizable** |

**Se adopta v4** pese al MAE superior: es la única versión sin error
sistemático en la franja horaria de mayor relevancia para la detección.

### Hallazgo con valor propio

> Un detector de anomalías basado en predicción, desplegado sobre datos
> ambientales, requiere recalibración continua de su línea base. Un modelo
> entrenado sobre un régimen estacional degrada su desempeño al operar sobre
> otro, y esa degradación se concentra en las horas de mayor concentración —que
> son precisamente las de mayor relevancia para la detección.

## Evaluación de detección sobre datos atacados

Hasta aquí sólo se ha medido **qué tan bien predice** el modelo. Falta lo que
decide si sirve: **si detecta ataques**.

### Procedimiento

El modelo entrenado no se toca. Lo que cambia es la serie de CCA en el conjunto
de prueba: se le inyectan ataques de bit flipping usando el mismo generador del
notebook 05.


**Punto clave:** las vecinas **no** se atacan. Es la premisa del escenario
multiestación: el adversario compromete un solo nodo. Las demás siguen
reportando la verdad, y por eso el modelo predice el valor legítimo aunque el
recibido esté manipulado.

### El umbral

El modelo produce un residuo continuo. Convertirlo en decisión binaria requiere
un corte, y ese corte **no puede calibrarse sobre datos atacados** — en
producción no se dispone de esa información.

Se calibra sobre los residuos del conjunto de entrenamiento limpio, tomando
percentiles. El percentil 95 significa: «marco lo que está en el 5% más
anómalo de lo observado en operación normal».

Es la misma limitación que `contamination` en IsolationForest, con una ventaja:
se puede barrer el percentil y reportar la curva completa de recall frente a
falsos positivos, en lugar de un solo punto.

### Qué se espera

Con σ del residuo ≈ 13 ppb:

| Ataque | Δ | Δ/σ | Pronóstico |
|---|---|---|---|
| bit 3 | 8 | 0.6× | No detectable |
| bit 4 | 16 | 1.2× | Dudoso |
| bit 5 | 32 | 2.5× | Detectable |
| bit 6 | 64 | 4.9× | Claro |

La referencia es el detector de Capa 1 (notebook 06), que sobre lecturas
aisladas alcanzaba 41.4% de recall en el bit 5 y 72.9% en el bit 6.

In [ ]:
from src.encoding.cayenne import encode_o3, decode_o3, flip_bit
from src.encoding.crypto import encrypt, decrypt
from src.attack.blind import APPSKEY, DEVADDR

def atacar_serie(valores, bit, tasa=0.05, seed=42):
    """Aplica bit flipping a una fraccion de las lecturas.

    Pasa por el cifrado real, no por un atajo aritmetico: el dataset
    depende de la implementacion que se defiende en la tesis.

    Devuelve (valores_atacados, etiquetas).
    """
    rng = np.random.default_rng(seed)
    v = valores.copy()
    etiq = np.zeros(len(v), dtype=int)

    for i in range(len(v)):
        if np.isnan(v[i]) or rng.random() >= tasa:
            continue
        trama = encrypt(encode_o3(int(v[i])), APPSKEY, DEVADDR, i + 1)
        atacada = flip_bit(trama, bit)
        v[i] = decode_o3(decrypt(atacada, APPSKEY, DEVADDR, i + 1))
        etiq[i] = 1

    return v, etiq

In [ ]:
# Residuos del modelo sobre entrenamiento limpio
pred_tr = Btr4 + modelo4.predict(Xtr4, verbose=0).flatten()
real_tr = Btr4 + ytr4
res_tr = np.abs(real_tr - pred_tr)

umbrales = {p: np.percentile(res_tr, p) for p in [90, 95, 97, 99]}
print("Umbrales calibrados sobre entrenamiento limpio:")
for p, u in umbrales.items():
    print(f"  p{p}: {u:6.2f} ppb")

In [ ]:
from sklearn.metrics import precision_score, recall_score, fbeta_score

# Serie limpia de CCA en el periodo de prueba
cca_te = mat_te['CCA'].copy()

res_det = []
for bit in [3, 4, 5, 6, 7]:
    atacada, etiq = atacar_serie(cca_te.values, bit, tasa=0.05, seed=42)

    mat_atk = mat_te.copy()
    mat_atk['CCA'] = atacada          # solo CCA se ataca

    Xa, ya, tsa, Ba, _, _ = construir_ventanas(
        mat_atk, 'CCA', ventana=24, mu=mu4, sigma=sg4,
        perfil=perf_te, predecir_desviacion=True)

    # Alinear etiquetas con las ventanas construidas
    idx = pd.Series(etiq, index=cca_te.index).reindex(pd.to_datetime(tsa)).values

    recibido = Ba + ya
    predicho = Ba + modelo4.predict(Xa, verbose=0).flatten()
    residuo = np.abs(recibido - predicho)

    for p, u in umbrales.items():
        marcado = (residuo > u).astype(int)
        res_det.append({
            'bit': bit, 'delta_ppb': 2**bit, 'percentil': p,
            'umbral_ppb': round(u, 1),
            'recall_%': round(recall_score(idx, marcado, zero_division=0)*100, 1),
            'precision_%': round(precision_score(idx, marcado, zero_division=0)*100, 1),
            'f2_%': round(fbeta_score(idx, marcado, beta=2, zero_division=0)*100, 1),
            'fp': int(((marcado == 1) & (idx == 0)).sum()),
        })

det = pd.DataFrame(res_det)
det.to_csv('../results/deteccion_lstm.csv', index=False)

det.pivot_table(index=['bit', 'delta_ppb'], columns='percentil',
                values='recall_%')

In [ ]:
print(det[det.bit.isin([4, 5, 6])][
    ['bit', 'percentil', 'umbral_ppb', 'recall_%', 'precision_%', 'f2_%', 'fp']
].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

bits = [3, 4, 5, 6, 7]
delta = [8, 16, 32, 64, 128]
capa1 = [4.3, 8.6, 41.4, 72.9, 97.1]      # DecisionTree, notebook 06
lstm  = [9.0, 20.5, 84.6, 98.7, 100.0]    # LSTM p95

x = np.arange(len(bits))
ancho = 0.38

fig, ax = plt.subplots(figsize=(9, 5))

b1 = ax.bar(x - ancho/2, capa1, ancho, label='Capa 1 — lecturas aisladas',
            color='steelblue', alpha=0.85)
b2 = ax.bar(x + ancho/2, lstm, ancho, label='LSTM — contexto espacial',
            color='seagreen', alpha=0.85)

for barras in (b1, b2):
    for b in barras:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 1.5,
                f'{b.get_height():.1f}', ha='center', fontsize=8)

ax.axvspan(1.5, 3.5, alpha=0.08, color='red')
ax.text(2.5, 108, 'zona explotable en Capa 1', ha='center',
        fontsize=9, color='crimson')

ax.set_xlabel('Bit atacado')
ax.set_ylabel('Recall sobre la clase atacada (%)')
ax.set_title('Detección por magnitud del ataque — CCA / O₃ / 2025')
ax.set_xticks(x)
ax.set_xticklabels([f'{b}\n{d} ppb' for b, d in zip(bits, delta)])
ax.set_ylim(0, 118)
ax.legend(loc='upper left')
ax.grid(alpha=0.25, axis='y')

plt.tight_layout()
plt.savefig('../docs/img/capa1_vs_lstm.png', dpi=150)
plt.show()

In [ ]:
daño = np.array([10.8, 11.4, 29.0, 100.0, 100.0])
nd_c1 = daño * (1 - np.array(capa1)/100)
nd_ls = daño * (1 - np.array(lstm)/100)

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(x, nd_c1, 'o-', lw=2.5, ms=8, color='crimson',
        label='Capa 1 — lecturas aisladas')
ax.plot(x, nd_ls, 's-', lw=2.5, ms=8, color='seagreen',
        label='LSTM — contexto espacial')
ax.fill_between(x, nd_ls, nd_c1, alpha=0.15, color='crimson')

ax.annotate(f'{nd_c1[3]:.1f}%', xy=(3, nd_c1[3]), xytext=(3.15, nd_c1[3]+2),
            fontsize=10, color='crimson', weight='bold')
ax.annotate(f'{nd_ls[1]:.1f}%', xy=(1, nd_ls[1]), xytext=(1.15, nd_ls[1]+2),
            fontsize=10, color='seagreen', weight='bold')

ax.set_xlabel('Bit atacado')
ax.set_ylabel('Daño no detectado (%)')
ax.set_title('Óptimo del adversario: daño que cambia la banda sin ser detectado')
ax.set_xticks(x)
ax.set_xticklabels([f'{b}\n{d} ppb' for b, d in zip(bits, delta)])
ax.legend()
ax.grid(alpha=0.25)

plt.tight_layout()
plt.savefig('../docs/img/daño_no_detectado.png', dpi=150)
plt.show()

In [ ]:
det = pd.read_csv('../results/deteccion_lstm.csv')

fig, ax = plt.subplots(figsize=(8, 5.5))

for bit, color, marca in [(5, 'darkorange', 'o'), (6, 'seagreen', 's')]:
    d = det[det.bit == bit].sort_values('percentil')
    ax.plot(d['precision_%'], d['recall_%'], marca + '-', lw=2, ms=8,
            color=color, label=f'bit {bit} ({2**bit} ppb)')
    for _, r in d.iterrows():
        ax.annotate(f"p{int(r.percentil)}",
                    (r['precision_%'], r['recall_%']),
                    textcoords='offset points', xytext=(7, -4), fontsize=8)

ax.axhline(41.4, ls=':', c='steelblue', lw=1.5)
ax.text(72, 43.5, 'Capa 1, bit 5 (41.4%)', fontsize=8, c='steelblue')

ax.set_xlabel('Precision (%)')
ax.set_ylabel('Recall (%)')
ax.set_title('Compromiso recall–precisión según el umbral')
ax.set_xlim(0, 80)
ax.set_ylim(0, 105)
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../docs/img/compromiso_percentil.png', dpi=150)
plt.show()

In [1]:
# Entrenamiento LSTM objetivo-especifico para todas las estaciones
# Reproduce el protocolo final usado con CCA (modelo v4).

import gc
import random
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import tensorflow as tf

RAIZ = Path.cwd().resolve()
if not (RAIZ / 'src').is_dir():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))

from src.ingest.rama import load_wide, to_long
from src.detect.windows import construir_ventanas, perfil_adaptativo
from src.detect.lstm import construir_modelo, entrenar

# Parametros identicos al experimento final de CCA.
VENTANA = 24
DIAS_PERFIL = 14
FRACCION_TRAIN = 0.80
EPOCAS = 60

long_33 = to_long(load_wide(RAIZ / 'data' / 'raw' / '2025O3.xls'), 'O3')
matriz_33 = (long_33.pivot(index='timestamp', columns='station', values='value')
                    .sort_index())
# Igual que en el experimento CCA: eliminar estaciones completamente vacias
# antes de construir las entradas. Si se conservaran, se inflaria la
# entrada a 72 features en vez de 66 (32 vecinas + 32 mascaras + 2 horas).
vacias_33 = matriz_33.columns[matriz_33.notna().sum() == 0].tolist()
matriz_33 = matriz_33.drop(columns=vacias_33)
estaciones_33 = matriz_33.columns.tolist()
corte_33 = int(len(matriz_33) * FRACCION_TRAIN)
mat_tr_33 = matriz_33.iloc[:corte_33]
mat_te_33 = matriz_33.iloc[corte_33:]

print(f'Estaciones sin datos y excluidas: {vacias_33}')
print(f'Estaciones objetivo: {len(estaciones_33)}')
print(f'Matriz: {matriz_33.shape[0]} horas x {len(estaciones_33)} estaciones')
print(f'Ventana: {VENTANA} h | perfil: {DIAS_PERFIL} dias | split: {FRACCION_TRAIN:.0%}')

resultados_33 = []
for numero, objetivo in enumerate(estaciones_33, start=1):
    # Semillas por objetivo: permite reproducir el modelo de cada estacion.
    semilla = 1000 + numero
    np.random.seed(semilla)
    random.seed(semilla)
    tf.random.set_seed(semilla)
    tf.keras.backend.clear_session()

    serie_objetivo = matriz_33[objetivo]
    perfil = perfil_adaptativo(serie_objetivo, dias=DIAS_PERFIL)
    perfil_tr = perfil.iloc[:corte_33]
    perfil_te = perfil.iloc[corte_33:]

    Xtr, ytr, ts_tr, Btr, mu, sigma = construir_ventanas(
        mat_tr_33, objetivo, ventana=VENTANA, perfil=perfil_tr,
        predecir_desviacion=True)
    Xte, yte, ts_te, Bte, _, _ = construir_ventanas(
        mat_te_33, objetivo, ventana=VENTANA, mu=mu, sigma=sigma,
        perfil=perfil_te, predecir_desviacion=True)

    if len(Xtr) == 0 or len(Xte) == 0:
        print(f'{objetivo}: omitida (sin ventanas validas)')
        continue

    modelo = construir_modelo(ventana=VENTANA, n_features=Xtr.shape[2])
    historia = entrenar(modelo, Xtr, ytr, val_frac=0.2,
                        epocas=EPOCAS, verbose=0)

    pred = Bte + modelo.predict(Xte, verbose=0).flatten()
    real = Bte + yte
    residuo = real - pred
    abs_residuo = np.abs(residuo)

    resultados_33.append({
        'estacion': objetivo,
        'n_train': len(Xtr),
        'n_test': len(Xte),
        'n_features': Xtr.shape[2],
        'mae_ppb': abs_residuo.mean(),
        'sesgo_ppb': residuo.mean(),
        'sigma_residuo_ppb': residuo.std(),
        'p95_residuo_ppb': np.percentile(abs_residuo, 95),
        'epocas_ejecutadas': len(historia.history['loss']),
    })
    print(f'{numero:02d}/{len(estaciones_33)} {objetivo}: '
          f'MAE={abs_residuo.mean():.2f} ppb | '
          f'sesgo={residuo.mean():+.2f} ppb | '
          f'sigma={residuo.std():.2f} ppb | '
          f'p95={np.percentile(abs_residuo, 95):.2f} ppb | '
          f'ventanas={len(Xte)}')

    del modelo, Xtr, ytr, Xte, yte, pred, real, residuo
    gc.collect()

resultados_33 = pd.DataFrame(resultados_33).sort_values('mae_ppb')
resultados_33.to_csv(RAIZ / 'results' / 'metricas_lstm_todas_estaciones.csv', index=False)
print('\nResumen ordenado por MAE:')
print(resultados_33.to_string(index=False, float_format=lambda x: f'{x:.2f}'))

I0000 00:00:1790130512.863914  110597 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1790130512.864347  110597 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790130512.894359  110597 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790130513.746794  110597 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

Estaciones sin datos y excluidas: ['COY', 'SFE', 'SJA']
Estaciones objetivo: 33
Matriz: 8760 horas x 33 estaciones
Ventana: 24 h | perfil: 14 dias | split: 80%


E0000 00:00:1790130515.309889  110597 cuda_platform.cc:57] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
=== Source Location Trace: === 
external/xla/xla/stream_executor/cuda/cuda_status.cc:50



01/33 ACO: MAE=7.67 ppb | sesgo=-6.58 ppb | sigma=7.03 ppb | p95=19.07 ppb | ventanas=59
02/33 AJM: MAE=12.55 ppb | sesgo=-0.04 ppb | sigma=16.40 ppb | p95=32.72 ppb | ventanas=1671
AJU: omitida (sin ventanas validas)
04/33 ATI: MAE=11.21 ppb | sesgo=+8.52 ppb | sigma=11.64 ppb | p95=29.02 ppb | ventanas=1681
05/33 BJU: MAE=9.02 ppb | sesgo=+6.13 ppb | sigma=10.83 ppb | p95=26.07 ppb | ventanas=478
06/33 CAM: MAE=8.28 ppb | sesgo=-1.17 ppb | sigma=12.33 ppb | p95=28.14 ppb | ventanas=1656
07/33 CCA: MAE=9.25 ppb | sesgo=+2.38 ppb | sigma=12.67 ppb | p95=27.14 ppb | ventanas=1684
CHO: omitida (sin ventanas validas)
09/33 CUA: MAE=11.05 ppb | sesgo=+0.17 ppb | sigma=14.49 ppb | p95=29.02 ppb | ventanas=1553
10/33 CUT: MAE=7.06 ppb | sesgo=+0.47 ppb | sigma=9.50 ppb | p95=19.47 ppb | ventanas=1604
11/33 FAC: MAE=8.79 ppb | sesgo=+0.71 ppb | sigma=11.62 ppb | p95=24.05 ppb | ventanas=1004
FAR: omitida (sin ventanas validas)
13/33 GAM: MAE=9.79 ppb | sesgo=+6.63 ppb | sigma=12.00 ppb | p95=

## Conclusiones del entrenamiento multiestación

El entrenamiento se realizó con el mismo protocolo V4 empleado para CCA: una ventana temporal de 24 horas, perfil adaptativo de 14 días, partición cronológica 80/20, predicción del residuo y uso de las estaciones vecinas como contexto espacial. Después de excluir las estaciones completamente vacías (`COY`, `SFE` y `SJA`), cada modelo utiliza 66 características: 32 valores normalizados de estaciones vecinas, 32 máscaras de disponibilidad y las codificaciones seno/coseno de la hora.

Se obtuvieron ventanas de entrenamiento y prueba válidas para 27 de las 33 estaciones. `AJU`, `CHO`, `FAR`, `IZT`, `MON` y `TAH` se omitieron porque no generaron ventanas válidas bajo este protocolo; esto debe reportarse como una limitación de cobertura, no como evidencia de buen o mal desempeño.

El desempeño es heterogéneo entre estaciones. Los valores de MAE van aproximadamente de 7 a 13 ppb, mientras que el percentil 95 del residuo absoluto va de aproximadamente 19 a 35 ppb. Esto indica que la variabilidad legítima y la capacidad de predicción dependen de la estación, por lo que no debe utilizarse un único umbral espacial para toda la red. Los sesgos también varían: algunas estaciones tienden a sobreestimar y otras a subestimar, aunque en varias el sesgo es cercano a cero.

Las métricas de estaciones con pocas ventanas de prueba, como `ACO` y `XAL`, deben interpretarse con cautela. Asimismo, los mensajes relacionados con CUDA indican únicamente que TensorFlow ejecutó el entrenamiento en CPU; no representan un fallo del experimento.

Estos resultados constituyen la línea base de predicción espacial. Todavía no miden la detección de bit flipping. El siguiente análisis debe aplicar ataques controlados a cada estación y evaluar cada lectura con el modelo y el umbral de esa misma estación. Con ello se calcularán el daño detectado, el daño no detectado, las falsas alarmas y su relación con las contingencias falsas y anuladas.

In [2]:
# Ataques multiestacion: entrenamiento V4 limpio + evaluacion exhaustiva.
from src.attack.blind import generar_dataset
BITS_ATAQUE = [3, 4, 5, 6, 7]
TAU_FASE1, TAU_FASE2 = 155, 200
resultados_ataque = []

for numero, objetivo in enumerate(estaciones_33, start=1):
    tf.keras.backend.clear_session()
    semilla = 1000 + numero
    np.random.seed(semilla); random.seed(semilla); tf.random.set_seed(semilla)
    perfil = perfil_adaptativo(matriz_33[objetivo], dias=DIAS_PERFIL)
    perfil_tr, perfil_te = perfil.iloc[:corte_33], perfil.iloc[corte_33:]
    Xtr, ytr, _, _, mu, sigma = construir_ventanas(mat_tr_33, objetivo, ventana=VENTANA, perfil=perfil_tr, predecir_desviacion=True)
    Xte, _, _, _, _, _ = construir_ventanas(mat_te_33, objetivo, ventana=VENTANA, mu=mu, sigma=sigma, perfil=perfil_te, predecir_desviacion=True)
    if len(Xtr) == 0 or len(Xte) == 0:
        print(f'{objetivo}: omitida (sin ventanas validas)')
        continue
    modelo = construir_modelo(ventana=VENTANA, n_features=Xtr.shape[2])
    entrenar(modelo, Xtr, ytr, val_frac=0.2, epocas=EPOCAS, verbose=0)
    Xc, yc, _, Bc, _, _ = construir_ventanas(mat_tr_33, objetivo, ventana=VENTANA, mu=mu, sigma=sigma, perfil=perfil_tr, predecir_desviacion=True)
    umbral = float(np.percentile(np.abs((Bc + yc) - (Bc + modelo.predict(Xc, verbose=0).flatten())), 95))
    serie_te = pd.DataFrame({'timestamp': mat_te_33.index, 'value': mat_te_33[objetivo].to_numpy()})
    dias_disp = serie_te.loc[serie_te.value.notna(), 'timestamp'].dt.normalize().nunique()

    for bit in BITS_ATAQUE:
        # Tasa 1.0 caracteriza todas las lecturas validas, sin muestreo aleatorio.
        atk = generar_dataset(serie_te, bit=bit, tasa=1.0, seed=semilla)
        mat_a = mat_te_33.copy()
        mat_a[objetivo] = atk.set_index('timestamp').received_value.reindex(mat_a.index)
        Xa, ya, tsa, Ba, _, _ = construir_ventanas(mat_a, objetivo, ventana=VENTANA, mu=mu, sigma=sigma, perfil=perfil_te, predecir_desviacion=True)
        if len(Xa) == 0:
            continue
        ai = atk.set_index('timestamp').reindex(pd.to_datetime(tsa))
        recibido = Ba + ya
        predicho = Ba + modelo.predict(Xa, verbose=0).flatten()
        marcado = np.abs(recibido - predicho) > umbral
        dano = ai['daño'].fillna(0).to_numpy().astype(int) == 1
        original, recibido_real = ai.original_value.to_numpy(), ai.received_value.to_numpy()
        fechas = pd.to_datetime(tsa).normalize()
        n_atk, n_dano = len(ai), int(dano.sum())
        n_dd, n_dnd = int((dano & marcado).sum()), int((dano & ~marcado).sum())
        falsa1 = (original < TAU_FASE1) & (recibido_real >= TAU_FASE1)
        anulada1 = (original >= TAU_FASE1) & (recibido_real < TAU_FASE1)
        # Fase 2: estos son cruces instantaneos de 200 ppb.
        # La condicion de permanencia durante una hora se evaluara aparte.
        cruce2_arriba = (original < TAU_FASE2) & (recibido_real >= TAU_FASE2)
        cruce2_abajo = (original >= TAU_FASE2) & (recibido_real < TAU_FASE2)
        dv = int(pd.Index(fechas[dano]).nunique())
        dnd = int(pd.Index(fechas[dano & ~marcado]).nunique())
        resultados_ataque.append({'estacion': objetivo, 'bit': bit, 'delta_ppb': 2**bit, 'umbral_p95_ppb': umbral,
            'n_ataques': n_atk, 'n_dano_banda': n_dano, 'n_detectados': int(marcado.sum()),
            'n_dano_detectado': n_dd, 'n_dano_no_detectado': n_dnd,
            'pct_dano': 100*n_dano/n_atk if n_atk else np.nan,
            'pct_dano_detectado': 100*n_dd/n_dano if n_dano else 0.0,
            'pct_dano_no_detectado': 100*n_dnd/n_dano if n_dano else 0.0,
            'dias_disponibles': dias_disp, 'dias_vulnerables': dv, 'dias_no_detectados': dnd,
            'pct_dias_vulnerables': 100*dv/dias_disp if dias_disp else np.nan,
            'pct_dias_dano_no_detectado': 100*dnd/dias_disp if dias_disp else np.nan,
            'falsas_fase1': int(falsa1.sum()), 'anuladas_fase1': int(anulada1.sum()),
            'cruces_fase2_arriba': int(cruce2_arriba.sum()), 'cruces_fase2_abajo': int(cruce2_abajo.sum())})
    print(f'{numero:02d}/{len(estaciones_33)} {objetivo}: p95={umbral:.2f} ppb | ventanas={len(Xte)}')
    del modelo, Xtr, ytr, Xte, Xc, yc, Xa, ya
    gc.collect()

ataques_33 = pd.DataFrame(resultados_ataque)
ataques_33.to_csv(RAIZ / 'results' / 'ataques_lstm_todas_estaciones.csv', index=False)
print('Resumen por bit:')
display(ataques_33.groupby('bit')[['pct_dias_vulnerables', 'pct_dias_dano_no_detectado', 'pct_dano_detectado', 'falsas_fase1', 'anuladas_fase1']].mean().round(2))


01/33 ACO: p95=13.90 ppb | ventanas=59
02/33 AJM: p95=32.17 ppb | ventanas=1671
AJU: omitida (sin ventanas validas)
04/33 ATI: p95=23.85 ppb | ventanas=1681
05/33 BJU: p95=22.23 ppb | ventanas=478
06/33 CAM: p95=21.28 ppb | ventanas=1656
07/33 CCA: p95=17.50 ppb | ventanas=1684
CHO: omitida (sin ventanas validas)
09/33 CUA: p95=34.00 ppb | ventanas=1553
10/33 CUT: p95=19.39 ppb | ventanas=1604
11/33 FAC: p95=21.23 ppb | ventanas=1004
FAR: omitida (sin ventanas validas)
13/33 GAM: p95=23.01 ppb | ventanas=561
14/33 HGM: p95=18.55 ppb | ventanas=1671
15/33 INN: p95=20.08 ppb | ventanas=1562
IZT: omitida (sin ventanas validas)
17/33 LLA: p95=13.37 ppb | ventanas=1563
18/33 LPR: p95=19.27 ppb | ventanas=1661
19/33 MER: p95=29.86 ppb | ventanas=1601
20/33 MGH: p95=14.79 ppb | ventanas=1385
MON: omitida (sin ventanas validas)
22/33 MPA: p95=19.47 ppb | ventanas=762
23/33 NEZ: p95=15.32 ppb | ventanas=1160
24/33 PED: p95=20.09 ppb | ventanas=1682
25/33 SAC: p95=19.10 ppb | ventanas=1430
26/33

,pct_dias_vulnerables,pct_dias_dano_no_detectado,pct_dano_detectado,falsas_fase1,anuladas_fase1
bit,,,,,
3,74.80,62.83,32.82,0.00,0.0
4,80.88,62.96,42.65,0.22,0.0
5,92.13,46.78,80.01,1.70,0.0
6,97.94,9.64,99.07,1.70,0.0
7,97.94,0.00,100.00,527.96,0.0


## Cómo interpretar el análisis de ataques multiestación

Esta celda evaluó una superficie de ataque exhaustiva: cada lectura válida del periodo de prueba fue atacada individualmente en los bits 3, 4, 5, 6 y 7. El modelo se entrenó únicamente con datos limpios y se aplicó después a la lectura atacada. Por ello, el experimento mide qué tan detectable es cada modificación cuando el atacante altera una estación objetivo y conserva limpias las 32 estaciones vecinas.

### Qué significan los porcentajes

| Métrica | Interpretación |
|---|---|
| Porcentaje de daño | Porcentaje de lecturas atacadas que cambiaron de banda NOM-172. |
| Porcentaje de daño detectado | De las lecturas que sí cambiaron de banda, porcentaje marcado por el LSTM. |
| Porcentaje de daño no detectado | De las lecturas que sí cambiaron de banda, porcentaje que pasó el detector. |
| Porcentaje de días vulnerables | Promedio por estación del porcentaje de días con al menos una lectura que cambió de banda. |
| Falsas de fase 1 | Lecturas que cruzaron 155 ppb hacia arriba después del flip. |
| Anuladas de fase 1 | Lecturas que estaban en 155 ppb o más y bajaron de ese umbral después del flip. |

El porcentaje de días vulnerables no equivale al porcentaje de días con contingencia. Aquí basta con que una lectura horaria de una estación cambie de banda. Al combinar 27 estaciones y todas las horas del periodo de prueba, es normal obtener valores altos, como 74.8% para el bit 3 y 97.9% para los bits 6 y 7. Esto no contradice el resultado de 08_proximidad_umbrales, donde se cuentan únicamente máximos diarios de la red que cruzan el umbral de contingencia.

### Resultado agregado multiestación

| Bit | Desplazamiento | Daño detectado | Daño no detectado por día | Lectura principal |
|---:|---:|---:|---:|---|
| 3 | 8 ppb | 32.82% | 62.83% | Poco visible; puede cambiar de banda cerca de un corte. |
| 4 | 16 ppb | 42.65% | 62.96% | Sigue siendo difícil de distinguir del ruido legítimo. |
| 5 | 32 ppb | 80.01% | 46.78% | Región intermedia: daño importante, pero todavía hay evasión. |
| 6 | 64 ppb | 99.07% | 9.64% | Muy dañino y casi siempre detectable. |
| 7 | 128 ppb | 100.00% | 0.00% | Daño máximo y detección completa. |

### Comparación con el experimento piloto de CCA

| Bit | Recall CCA (p95) | Daño detectado multiestación |
|---:|---:|---:|
| 3 | 9.0% | 32.82% |
| 4 | 20.5% | 42.65% |
| 5 | 84.6% | 80.01% |
| 6 | 98.7% | 99.07% |
| 7 | 100.0% | 100.00% |

Estas columnas no son idénticas: CCA reporta recall de todos los mensajes atacados en un experimento con tasa de ataque del 5%, mientras que la tabla multiestación reporta detección condicionada a que el ataque haya cambiado de banda, usando una superficie exhaustiva. La comparación es orientativa, no una prueba estadística directa. Aun así, ambas muestran la misma frontera cualitativa: los bits 3 y 4 son los más difíciles de detectar, el bit 5 es intermedio y los bits 6 y 7 son evidentes.

### Diferencia con las contingencias del notebook 08

El notebook 08 ya calculó la operación correcta a nivel de red: toma el máximo diario, aplica el flip respetando el estado original del bit y cuenta si se fabrica o se anula una contingencia. Allí se obtuvieron, para el análisis anual previo, 4 días vulnerables al bit 3, 17 al bit 4 y 63 al bit 5; además, se identificaron contingencias reales que podían anularse con los bits 3, 4 y 5. Esos números no deben sustituirse por los porcentajes altos de esta celda: responden a unidades de análisis diferentes.

Por tanto, el resultado multiestación sirve para cuantificar la detectabilidad del daño local, mientras que el análisis de 08_proximidad_umbrales sirve para cuantificar el impacto operativo sobre la contingencia de la red. La gráfica final debe combinar ambos análisis sólo después de recalcular el análisis de red con el umbral confirmado de 155 ppb y, si se desea medir fase 2, aplicar también la condición de permanencia durante una hora.
